# Visiopharm Segmentation to GeoJSON Converter

Standalone tool to convert Visiopharm segmentation TIFF files to GeoJSON format.

## Features:
- Extracts cell segmentation masks from Visiopharm TIFF files
- Converts cell boundaries to polygon geometries
- Handles complex shapes with holes
- Outputs GeoJSON FeatureCollection
- Optionally outputs CSV with polygon vertices

## Usage:
1. Set input/output paths in the configuration cell
2. Run all cells
3. GeoJSON files will be saved to output directory

In [ ]:
import tifffile as tiff
from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import os
import pandas as pd
import json

from tqdm import tqdm
from skimage.measure import regionprops, find_contours, approximate_polygon, label

print("All libraries imported successfully!")

## Configuration

Set your input and output paths here.

In [ ]:
# ============= USER CONFIGURATION =============

# Input path: Directory containing Visiopharm segmentation TIFF files
home_path = os.path.expanduser("~")
visio_input_dir = Path(home_path) / 'ext_hd_sammy' / 'data' / 'COMET' / 'visio_output' / '6_Samples_Trial' / '6_Samples_mld_xml'

# Optional: Path to original images (for coordinate validation/overlay)
# Set to None if not needed
original_image_dir = Path(home_path) / 'ext_hd_sammy' / 'data' / 'COMET' / 'original'

# Output directories
output_geojson_dir = Path(home_path) / 'ext_hd_sammy' / 'projects' / 'out' / 'out_comet_geojson'
output_csv_dir = Path(home_path) / 'ext_hd_sammy' / 'projects' / 'out' / 'out_comet_csv_polygons'

# Processing options
POLYGON_SIMPLIFICATION_TOLERANCE = 1.5  # Higher = fewer vertices, lower = more detail
EXPORT_CSV = True  # Set to False if you only want GeoJSON
FLIP_X_AXIS = True  # Set to True to flip x-coordinates to match original images

# ============================================

# Create output directories
os.makedirs(output_geojson_dir, exist_ok=True)
if EXPORT_CSV:
    os.makedirs(output_csv_dir, exist_ok=True)

print(f"Input directory: {visio_input_dir}")
print(f"GeoJSON output: {output_geojson_dir}")
if EXPORT_CSV:
    print(f"CSV output: {output_csv_dir}")
print(f"\nPolygon simplification tolerance: {POLYGON_SIMPLIFICATION_TOLERANCE}")
print(f"Flip X-axis: {FLIP_X_AXIS}")

## Core Functions

In [ ]:
def _signed_area(ring_xy):
    """Calculate signed area of a polygon ring."""
    x = ring_xy[:, 0]
    y = ring_xy[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)


def _ensure_closed(r):
    """Ensure polygon ring is closed (first point == last point)."""
    if not np.allclose(r[0], r[-1]):
        r = np.vstack([r, r[0]])
    return r


def roi_to_rings_xy(prop, image_width, simplify_tol=1.5, flip_x=True):
    """Convert a regionprops object to polygon rings with proper orientation.
    
    Args:
        prop: regionprops object from scikit-image
        image_width: Width of the image (needed for x-axis flipping)
        simplify_tol: Tolerance for polygon simplification (higher = fewer vertices)
        flip_x: Whether to flip x-coordinates to match coordinate systems
    
    Returns:
        List of polygon rings: [exterior, hole1, hole2, ...]
        Each ring is a list of [x, y] coordinates
    """
    # Find contours in the binary mask
    cnts = find_contours(prop.image.astype(float), 0.5)
    if not cnts:
        return []
    
    # Get bounding box to translate coordinates to image space
    y0, x0, y1, x1 = prop.bbox

    rings = []
    for cnt in cnts:
        # Translate contour coordinates to image space
        cnt[:, 0] += y0  # rows -> y
        cnt[:, 1] += x0  # cols -> x
        
        # Convert to (x, y) format
        ring_xy = np.c_[cnt[:, 1], cnt[:, 0]]
        
        # Simplify polygon to reduce vertex count
        ring_xy = approximate_polygon(ring_xy, tolerance=simplify_tol)
        
        # Ensure polygon is closed
        ring_xy = _ensure_closed(ring_xy)
        
        # Flip x-coordinates if requested
        if flip_x:
            ring_xy[:, 0] = (image_width - 1) - ring_xy[:, 0]
        
        rings.append(ring_xy)

    # Identify exterior ring (largest area) vs holes
    areas = [abs(_signed_area(r)) for r in rings]
    ext_idx = int(np.argmax(areas))
    exterior = rings[ext_idx]
    holes = [rings[i] for i in range(len(rings)) if i != ext_idx]

    # Ensure proper orientation: counter-clockwise for exterior, clockwise for holes
    if _signed_area(exterior) < 0:
        exterior = exterior[::-1]
    
    oriented_holes = []
    for h in holes:
        if _signed_area(h) > 0:
            h = h[::-1]
        oriented_holes.append(h)

    # Return as list: [exterior, hole1, hole2, ...]
    rings_xy = [exterior.tolist()] + [h.tolist() for h in oriented_holes]
    return rings_xy


print("Polygon functions defined successfully!")

In [ ]:
def visiopharm_to_geojson(visio_tiff_path, image_width, simplify_tol=1.5, flip_x=True):
    """Convert Visiopharm segmentation TIFF to GeoJSON FeatureCollection.
    
    Args:
        visio_tiff_path: Path to Visiopharm segmentation TIFF file
        image_width: Width of the original image
        simplify_tol: Polygon simplification tolerance
        flip_x: Whether to flip x-coordinates
    
    Returns:
        Dictionary containing GeoJSON FeatureCollection
    """
    # Read segmentation mask
    v_img = tiff.imread(visio_tiff_path)
    
    # Extract channel 0 and create labeled regions
    single_cell = label(v_img[0, ...] > 0)
    
    # Extract region properties
    props = regionprops(single_cell)
    
    print(f"Found {len(props)} cells in {Path(visio_tiff_path).name}")
    
    # Convert each cell to a GeoJSON feature
    features = []
    for prop in tqdm(props, desc="Converting cells to polygons"):
        rings_xy = roi_to_rings_xy(prop, image_width, simplify_tol=simplify_tol, flip_x=flip_x)
        
        if not rings_xy:
            continue
        
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": rings_xy
            },
            "properties": {
                "label": int(prop.label),
                "area_px": int(prop.area),
                "centroid_x": float(prop.centroid[1]),
                "centroid_y": float(prop.centroid[0]),
                "bbox": [int(v) for v in prop.bbox]  # [min_row, min_col, max_row, max_col]
            }
        }
        features.append(feature)
    
    # Create FeatureCollection
    feature_collection = {
        "type": "FeatureCollection",
        "features": features
    }
    
    return feature_collection


def extract_vertices_to_dataframe(feature_collection):
    """Extract polygon vertices to a pandas DataFrame.
    
    Args:
        feature_collection: GeoJSON FeatureCollection dictionary
    
    Returns:
        DataFrame with columns: vertex_x, vertex_y, cell_id
    """
    data = []
    
    for feature in feature_collection["features"]:
        cell_id = feature["properties"]["label"]
        coords = feature["geometry"]["coordinates"][0]  # Exterior ring only
        
        # Extract x and y coordinates
        vertex_x = [coord[0] for coord in coords]
        vertex_y = [coord[1] for coord in coords]
        
        data.append({
            "vertex_x": vertex_x,
            "vertex_y": vertex_y,
            "cell_id": cell_id
        })
    
    return pd.DataFrame(data)


print("Conversion functions defined successfully!")

## Process Visiopharm Files

In [ ]:
# Find all TIFF files in input directory
visio_files = sorted([f for f in os.listdir(visio_input_dir) if f.endswith('.tif')])

print(f"Found {len(visio_files)} Visiopharm segmentation files:")
for f in visio_files:
    print(f"  - {f}")

In [ ]:
# Determine image width (needed for x-axis flipping)
# Option 1: Read from original image if available
# Option 2: Read from segmentation mask itself

sample_visio_file = visio_input_dir / visio_files[0]
sample_img = tiff.imread(sample_visio_file)
image_width = sample_img.shape[2] if len(sample_img.shape) > 2 else sample_img.shape[1]

print(f"Detected image width: {image_width} pixels")
print(f"Sample image shape: {sample_img.shape}")

# Verify this is correct
if original_image_dir and original_image_dir.exists():
    orig_files = sorted([f for f in os.listdir(original_image_dir) if f.endswith('.tiff') or f.endswith('.ome.tiff')])
    if orig_files:
        sample_orig = tiff.imread(original_image_dir / orig_files[0])
        orig_width = sample_orig.shape[2] if len(sample_orig.shape) > 2 else sample_orig.shape[1]
        print(f"Original image width: {orig_width} pixels")
        if orig_width != image_width:
            print(f"WARNING: Image width mismatch! Using original image width: {orig_width}")
            image_width = orig_width

In [ ]:
# Process all Visiopharm files
print(f"\nProcessing {len(visio_files)} files...\n")

for visio_file in visio_files:
    print(f"\n{'='*60}")
    print(f"Processing: {visio_file}")
    print(f"{'='*60}")
    
    visio_path = visio_input_dir / visio_file
    base_name = visio_file.replace('.tif', '')
    
    # Convert to GeoJSON
    feature_collection = visiopharm_to_geojson(
        visio_path,
        image_width=image_width,
        simplify_tol=POLYGON_SIMPLIFICATION_TOLERANCE,
        flip_x=FLIP_X_AXIS
    )
    
    # Save GeoJSON
    geojson_path = output_geojson_dir / f"{base_name}_masks.geojson"
    with open(geojson_path, 'w') as f:
        json.dump(feature_collection, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved GeoJSON: {geojson_path}")
    print(f"  ({len(feature_collection['features'])} cells)")
    
    # Optionally save CSV
    if EXPORT_CSV:
        df = extract_vertices_to_dataframe(feature_collection)
        csv_path = output_csv_dir / f"{base_name}_vertices.csv"
        df.to_csv(csv_path, index=False)
        print(f"✓ Saved CSV: {csv_path}")
        print(f"  ({len(df)} cells)")

print(f"\n{'='*60}")
print("All files processed successfully!")
print(f"{'='*60}")

## Validation and Visualization (Optional)

Visualize a sample of converted cells to verify correctness.

In [ ]:
# Load a sample GeoJSON for validation
sample_geojson = output_geojson_dir / f"{visio_files[0].replace('.tif', '')}_masks.geojson"

with open(sample_geojson) as f:
    fc = json.load(f)

print(f"Loaded: {sample_geojson.name}")
print(f"Type: {fc['type']}")
print(f"Number of features: {len(fc['features'])}")
print(f"\nSample feature:")
print(json.dumps(fc['features'][0], indent=2))

In [ ]:
# Visualize sample polygons overlaid on image (if original images available)
import random
from matplotlib.patches import Polygon as MplPolygon

def show_overlay(image, feature, ax=None):
    """Show polygon overlay on image."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    ax.imshow(image, cmap="gray")
    
    coords = feature["geometry"]["coordinates"]
    
    # Draw exterior ring
    ext = np.array(coords[0])
    ax.add_patch(MplPolygon(ext, fill=False, edgecolor='red', linewidth=2))
    
    # Draw holes (if any)
    for hole in coords[1:]:
        ax.add_patch(MplPolygon(np.array(hole), fill=False, edgecolor='blue', 
                                linestyle='--', linewidth=1.5))
    
    label = feature['properties']['label']
    area = feature['properties']['area_px']
    ax.set_title(f"Cell ID: {label} | Area: {area} px²")
    ax.axis("off")
    return ax


if original_image_dir and original_image_dir.exists():
    # Load corresponding original image
    orig_file_name = visio_files[0].replace('.tif', '.ome.tiff')
    orig_path = original_image_dir / orig_file_name
    
    if orig_path.exists():
        orig_img = tiff.imread(orig_path)
        
        # Flip if necessary
        if FLIP_X_AXIS:
            display_img = np.flip(orig_img, axis=2)
        else:
            display_img = orig_img
        
        # Show first channel (usually DAPI)
        display_img = display_img[0, ...] if len(display_img.shape) > 2 else display_img
        
        # Plot random sample of cells
        num_samples = min(6, len(fc['features']))
        sample_features = random.sample(fc['features'], k=num_samples)
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, feat in enumerate(sample_features):
            show_overlay(display_img, feat, ax=axes[i])
        
        plt.tight_layout()
        plt.show()
        
        print(f"Displayed {num_samples} sample cells with polygon overlays")
    else:
        print(f"Original image not found: {orig_path}")
else:
    print("Original image directory not available for visualization")

## Summary Statistics

In [ ]:
# Collect statistics from all processed files
stats_data = []

for visio_file in visio_files:
    base_name = visio_file.replace('.tif', '')
    geojson_path = output_geojson_dir / f"{base_name}_masks.geojson"
    
    with open(geojson_path) as f:
        fc = json.load(f)
    
    num_cells = len(fc['features'])
    areas = [f['properties']['area_px'] for f in fc['features']]
    
    stats_data.append({
        'file': visio_file,
        'num_cells': num_cells,
        'mean_area': np.mean(areas),
        'median_area': np.median(areas),
        'min_area': np.min(areas),
        'max_area': np.max(areas)
    })

stats_df = pd.DataFrame(stats_data)
print("\nProcessing Summary:")
print("="*80)
print(stats_df.to_string(index=False))
print("="*80)
print(f"\nTotal cells processed: {stats_df['num_cells'].sum()}")
print(f"Average cells per file: {stats_df['num_cells'].mean():.1f}")

In [ ]:
print("\n" + "="*80)
print("CONVERSION COMPLETE!")
print("="*80)
print(f"\nGeoJSON files saved to: {output_geojson_dir}")
if EXPORT_CSV:
    print(f"CSV files saved to: {output_csv_dir}")
print(f"\nTotal files processed: {len(visio_files)}")
print(f"Total cells converted: {stats_df['num_cells'].sum()}")